# W2-D2: Root Cause Analysis — Graph, Causal & LLM-augmented

**Pipeline:** Alert Cluster → Graph+Temporal Scoring → Keyword Retrieval → kNN Classifier → RCA Output

**Phương pháp:** Graph Traversal (PageRank reverse + Sink detection + Timestamp) + Keyword-based Retrieval (kNN-style)

## Cell 1: Load Data

In [4]:
import json
import os
import networkx as nx
from rca import (
    build_service_graph,
    rca_graph_temporal,
    retrieve_similar_incidents,
    classify_from_similar,
    validate_output,
    run_rca_pipeline,
    compute_similarity,
)

# Load datasets
dataset_dir = 'dataset'

with open(os.path.join(dataset_dir, 'cluster_summary.json'), 'r', encoding='utf-8') as f:
    cluster_summary = json.load(f)

with open(os.path.join(dataset_dir, 'alerts_sample.jsonl'), 'r', encoding='utf-8') as f:
    alerts = [json.loads(line) for line in f if line.strip()]

with open(os.path.join(dataset_dir, 'services.json'), 'r', encoding='utf-8') as f:
    services_data = json.load(f)

with open(os.path.join(dataset_dir, 'incidents_history.json'), 'r', encoding='utf-8') as f:
    incidents_data = json.load(f)

print(f'Clusters: {cluster_summary["output_clusters"]}')
print(f'Alerts: {len(alerts)}')
print(f'Incidents history: {len(incidents_data["incidents"])}')
print(f'Services: {len(services_data["services"])}')
print(f'Stores: {len(services_data["stores"])}')
print(f'Edges: {len(services_data["edges"])}')

Clusters: 3
Alerts: 20
Incidents history: 29
Services: 10
Stores: 4
Edges: 17


## Cell 2: Build Service Graph & Visualize

In [5]:
# Build directed graph
graph = build_service_graph(services_data)
print(f'Service graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print()

# In danh sách nodes + degree
print('Node | In-Degree | Out-Degree')
print('-' * 45)
for node in sorted(graph.nodes()):
    in_deg = graph.in_degree(node)
    out_deg = graph.out_degree(node)
    print(f'{node:25s} | {in_deg:9d} | {out_deg:10d}')

print()
print('Edges (caller -> callee):')
for u, v, data in graph.edges(data=True):
    print(f'  {u} -> {v}  [{data.get("type", "http")}]')

Service graph: 14 nodes, 17 edges

Node | In-Degree | Out-Degree
---------------------------------------------
auth-svc                  |         1 |          0
cart-redis                |         1 |          0
cart-svc                  |         1 |          2
catalog-db                |         4 |          0
catalog-svc               |         2 |          2
checkout-svc              |         1 |          4
edge-lb                   |         0 |          4
inventory-svc             |         1 |          1
kafka-events              |         1 |          0
notification-svc          |         1 |          1
payment-svc               |         1 |          1
payments-db               |         1 |          0
recommender-svc           |         1 |          1
search-svc                |         1 |          1

Edges (caller -> callee):
  edge-lb -> auth-svc  [http]
  edge-lb -> catalog-svc  [http]
  edge-lb -> search-svc  [http]
  edge-lb -> checkout-svc  [http]
  checkout-svc -> c

## Cell 3: RCA cho cluster chính (c-000-000)

In [6]:
# Cluster chinh: c-000-000 (18 alerts, 5 services)
cluster_main = cluster_summary['clusters'][0]
print(f'Cluster: {cluster_main["cluster_id"]}')
print(f'Services: {cluster_main["services"]}')
print(f'Alert count: {cluster_main["alert_count"]}')
print(f'Max severity: {cluster_main["max_severity"]}')
print(f'Time range: {cluster_main["time_range"]}')
print()

# ---- Step 1: Graph + Temporal RCA ----
print('=' * 50)
print('STEP 1: Graph + Temporal Scoring')
print('=' * 50)

# Xay dung alert subgraph
alert_nodes = [s for s in cluster_main['services'] if s in graph]
subgraph = graph.subgraph(alert_nodes).copy()
print(f'\nAlert subgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges')
print('Subgraph edges:')
for u, v in subgraph.edges():
    print(f'  {u} -> {v}')

# Tim sink (out_degree = 0)
print('\nSink analysis (out_degree = 0 in alert subgraph):')
for node in alert_nodes:
    od = subgraph.out_degree(node)
    sink_label = ' <-- SINK (root cause candidate)' if od == 0 else ''
    print(f'  {node}: out_degree={od}{sink_label}')

# Top candidates
candidates = rca_graph_temporal(cluster_main, alerts, graph)
print('\nTop-3 candidates (combined score):')
for svc, score in candidates:
    print(f'  {svc}: {score:.3f}')

Cluster: c-000-000
Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
Alert count: 18
Max severity: crit
Time range: ['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z']

STEP 1: Graph + Temporal Scoring

Alert subgraph: 5 nodes, 4 edges
Subgraph edges:
  edge-lb -> checkout-svc
  checkout-svc -> cart-svc
  checkout-svc -> payment-svc
  checkout-svc -> notification-svc

Sink analysis (out_degree = 0 in alert subgraph):
  cart-svc: out_degree=0 <-- SINK (root cause candidate)
  checkout-svc: out_degree=3
  edge-lb: out_degree=1
  notification-svc: out_degree=0 <-- SINK (root cause candidate)
  payment-svc: out_degree=0 <-- SINK (root cause candidate)

Top-3 candidates (combined score):
  payment-svc: 0.775
  checkout-svc: 0.504
  cart-svc: 0.441


In [7]:
# ---- Step 2: Retrieve similar incidents ----
print('=' * 50)
print('STEP 2: Keyword-based Retrieval')
print('=' * 50)

similar = retrieve_similar_incidents(cluster_main, incidents_data['incidents'])
print(f'\nFound {len(similar)} similar incidents (score >= 0.2):\n')
for inc, sim in similar:
    print(f'  {inc["id"]} (similarity={sim:.2f})')
    print(f'    root_cause: {inc["root_cause_service"]} ({inc["root_cause_class"]})')
    print(f'    services: {inc["services_involved"]}')
    print(f'    summary: {inc["summary"][:100]}...')
    print()

STEP 2: Keyword-based Retrieval

Found 3 similar incidents (score >= 0.2):

  INC-2025-11-08 (similarity=1.00)
    root_cause: payment-svc (connection_pool_exhaustion)
    services: ['payment-svc', 'payments-db', 'checkout-svc']
    summary: Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream checkout cas...

  INC-2026-03-20 (similarity=1.00)
    root_cause: edge-lb (ddos)
    services: ['edge-lb', 'checkout-svc', 'payment-svc']
    summary: Volumetric DDoS 5x normal traffic. Edge-lb saturate, all upstream visible degraded....

  INC-2025-09-05 (similarity=0.80)
    root_cause: payment-svc (connection_pool_exhaustion)
    services: ['payment-svc', 'payments-db']
    summary: Payment timeout 100%. Deploy v2.6 leak DB connection — pool from 50 → 50 hold, never return....



In [8]:
# ---- Step 3: Classify from kNN ----
print('=' * 50)
print('STEP 3: kNN Classification')
print('=' * 50)

classification = classify_from_similar(similar)
print(f'\nRoot cause class: {classification["class"]}')
print(f'Actions:')
for i, action in enumerate(classification['actions'], 1):
    print(f'  {i}. {action}')
print(f'\nReasoning: {classification["reasoning"]}')
print(f'Similar incidents: {classification["similar_ids"]}')

STEP 3: kNN Classification

Root cause class: connection_pool_exhaustion
Actions:
  1. Rollback to v3.1
  2. Scale pool 50 → 100 cushion
  3. Add pool monitor alert > 80%

Reasoning: Matched historical incident INC-2025-11-08 (similarity=1.00): Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream checkout cascade. Notification queue backed up.
Similar incidents: ['INC-2025-11-08', 'INC-2026-03-20', 'INC-2025-09-05']


## Cell 4: Run Full Pipeline & Write Output

In [9]:
# Run full pipeline cho tat ca clusters
output = run_rca_pipeline(cluster_summary, alerts, graph, incidents_data)

# Write output
os.makedirs('results', exist_ok=True)
with open('results/rca_output.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'\nOutput written to results/rca_output.json')
print(f'Clusters analyzed: {output["clusters_analyzed"]}')


Analyzing cluster: c-000-000
  Services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
  Alert count: 18
  Max severity: crit

  [Graph+Temporal] Top candidates:
    payment-svc: 0.775
    checkout-svc: 0.504
    cart-svc: 0.441

  [Retrieval] Top similar incidents:
    INC-2025-11-08 (sim=1.00): connection_pool_exhaustion
    INC-2026-03-20 (sim=1.00): ddos
    INC-2025-09-05 (sim=0.80): connection_pool_exhaustion

  [Classifier] class=connection_pool_exhaustion
  [Classifier] actions=['Rollback to v3.1', 'Scale pool 50 → 100 cushion', 'Add pool monitor alert > 80%']

  ✓ Root cause: payment-svc
  ✓ Class: connection_pool_exhaustion
  ✓ Confidence: 0.77

Analyzing cluster: c-000-001
  Services: ['recommender-svc']
  Alert count: 1
  Max severity: warn

  [Graph+Temporal] Top candidates:
    recommender-svc: 1.000

  [Retrieval] Top similar incidents:
    INC-2025-08-02 (sim=0.60): memory_leak
    INC-2025-10-28 (sim=0.60): model_drift
    INC-2026-03-07 (

## Cell 5: Summary Table

In [10]:
# Bang tong hop ket qua
print(f'{"Cluster":15s} | {"Root Cause":20s} | {"Class":35s} | {"Confidence":>10s} | {"Method"}')
print('-' * 110)
for r in output['results']:
    print(f'{r["cluster_id"]:15s} | {r["root_cause"]:20s} | {r["class"]:35s} | {r["confidence"]:>10.2f} | {r["method"]}')

print()
print('Top-3 candidates per cluster:')
for r in output['results']:
    print(f'\n  [{r["cluster_id"]}]')
    for svc, score in r['graph_top3']:
        print(f'    {svc}: {score:.2f}')

print()
print('Similar historical incidents per cluster:')
for r in output['results']:
    print(f'\n  [{r["cluster_id"]}] -> {r["similar_incidents"]}')

Cluster         | Root Cause           | Class                               | Confidence | Method
--------------------------------------------------------------------------------------------------------------
c-000-000       | payment-svc          | connection_pool_exhaustion          |       0.77 | graph+knn
c-000-001       | recommender-svc      | memory_leak                         |       1.00 | graph+knn
c-000-002       | search-svc           | n_plus_1                            |       1.00 | graph+knn

Top-3 candidates per cluster:

  [c-000-000]
    payment-svc: 0.77
    checkout-svc: 0.50
    cart-svc: 0.44

  [c-000-001]
    recommender-svc: 1.00

  [c-000-002]
    search-svc: 1.00

Similar historical incidents per cluster:

  [c-000-000] -> ['INC-2025-11-08', 'INC-2026-03-20', 'INC-2025-09-05']

  [c-000-001] -> ['INC-2025-08-02', 'INC-2025-10-28', 'INC-2026-03-07']

  [c-000-002] -> ['INC-2026-01-29', 'INC-2026-05-25', 'INC-2025-12-01']
